# Capítulo VI – Programación y Simulación
## Análisis Cuantitativo con R: Matemáticas, Estadística y Econometría

**Autores del libro:** Daniel Liviano Solís · Maria Pujol Jover
**Editorial:** UOC · Primera edición digital: junio 2017
**Fuente:** Capítulo VI, págs. 215-236

---

### Objetivos de aprendizaje
Al finalizar este notebook serás capaz de:
1. Generar valores aleatorios reproducibles en R con `set.seed()` y `sample()`.
2. Comprobar mediante simulación de Monte Carlo el sesgo del estimador de la varianza muestral.
3. Verificar empíricamente el Teorema Central del Límite (TCL) simulando la distribución de la media muestral.
4. Escribir una función propia (`tcl()`) que generalice una simulación a cualquier población de entrada.

### Nota sobre este notebook
Este notebook estaba previamente escrito en **Python** como traducción del capítulo original.
Se ha **reescrito íntegramente en R** (kernel `ir`) para que el código corresponda exactamente
al lenguaje y a los ejemplos del libro, incluyendo semillas (`set.seed`) y salidas idénticas a las del texto.

### Cómo usar este notebook
Cada apartado sigue el **mismo número de sección del libro** (p. ej. `4.2` = Aproximación de una
distribución totalmente aleatoria). Ejecuta las celdas en orden: varias reutilizan objetos
(`pob`, `pob.2`, `pob.3`, `tcl`) creados en celdas anteriores. La sección 4 requiere el paquete `moments`.

---
# 1. Motivación (pág. 215)

En estadística, a menudo no se dispone de datos empíricos para validar teorías u obtener
información estadística. En ese caso, la **simulación** es una alternativa muy eficaz: consiste
en generar valores aleatorios y tratarlos como si fueran datos reales extraídos de una
población o generados por un experimento real. Los datos generados de esta manera se
denominan **simulados** o **sintéticos**.

La simulación tiene diversas aplicaciones: estimación de probabilidades y parámetros
estadísticos (media, varianza), verificación del supuesto de normalidad, estimación de sesgos
y estimación de distribuciones de probabilidad para la realización de inferencia.

Este capítulo se estructura en tres aplicaciones empíricas:
1. Generación de datos aleatorios con R.
2. Estudio del sesgo del estimador de la varianza muestral.
3. Análisis del Teorema Central del Límite mediante simulación.

---
# 2. Generación de valores aleatorios (pág. 216-218)

### La función `set.seed(seed)`
Se coloca antes de una instrucción en la que se generan valores aleatorios. Fijando un valor
arbitrario de `seed`, nos aseguramos de obtener siempre los mismos valores en la instrucción.

### La función `sample(x, size, replace, prob)`
Extrae aleatoriamente valores del vector `x` hasta completar un vector de longitud `size`.
`replace` indica si las extracciones son con reemplazamiento (`TRUE`) o sin él (`FALSE`).
`prob` es un vector opcional de probabilidades de extracción; si no se especifica, todos los
elementos de `x` son equiprobables.

In [ ]:
# ---------------------------------------------------------------
# 2a Muestreo CON reemplazo: lanzamiento de una moneda (pag. 216)
# Espacio muestral: {C, X} (cara, cruz)
# ---------------------------------------------------------------
sample(c("C", "X"), 10, replace = TRUE)
# [1] "X" "C" "X" "C" "X" "X" "C" "X" "C" "X"

In [ ]:
# ---------------------------------------------------------------
# 2b Muestreo SIN reemplazo: 3 numeros del 1 al 20 (pag. 217)
# set.seed() fija la semilla para que el resultado sea replicable
# ---------------------------------------------------------------
set.seed(5)
sample(1:20, 3, replace = FALSE)
# [1]  5 14 17

In [ ]:
# ---------------------------------------------------------------
# 2c Figura VI.1: histograma de una Normal(0,1) con densidad teorica (pag. 217)
# rnorm(n, mu, sigma) genera n valores aleatorios de una N(mu, sigma)
# curve(add=TRUE) superpone la funcion de densidad teorica dnorm()
# ---------------------------------------------------------------
v <- rnorm(1000, 0, 1)
hist(v, probability = TRUE, col = "grey", main = "")
curve(dnorm(x, 0, 1), add = TRUE, lwd = 4)

In [ ]:
# ---------------------------------------------------------------
# 2d Figura VI.2: histograma de una Exponencial(lambda=1) (pag. 218)
# rexp(n, rate) genera n valores aleatorios de una Exponencial
# ---------------------------------------------------------------
v <- rexp(100, 1)
hist(v, probability = TRUE, col = "grey", main = "")
curve(dexp(x, 1), add = TRUE, lwd = 4)

---
# 3. Aplicación empírica: sesgo del estimador de la varianza muestral (pág. 218-223)

### Conceptos clave
Para una población de N individuos con media poblacional μ, la varianza poblacional es:

σ² = Σ(xᵢ − μ)² / N

Con una muestra de n observaciones y media muestral x̄, la **varianza muestral sesgada** es:

σ̂*² = Σ(xᵢ − x̄)² / n

Este estimador es sesgado: su valor esperado no es σ², sino **E[σ̂*²] = (n−1)/n · σ²**.
La corrección del denominador (n−1) produce la **varianza muestral insesgada**:

σ̂² = Σ(xᵢ − x̄)² / (n−1), con E[σ̂²] = σ²

### Estrategia de simulación (5 pasos del libro)
1. Generar K=100.000 muestras aleatorias de N(0,1) para tamaños n=2,...,10.
2. Calcular la varianza muestral sesgada de cada muestra y almacenarla en una matriz V (K × 9).
3. Calcular la media aritmética de cada columna de V.
4. Comparar los cocientes observados con el valor teórico (n−1)/n.
5. Representar gráficamente el resultado con un diagrama de barras.

In [ ]:
# ---------------------------------------------------------------
# 3a Paso 1: definir K (numero de muestras) y N (tamanio muestral maximo) (pag. 221)
# ---------------------------------------------------------------
K <- 100000
N <- 10

In [ ]:
# ---------------------------------------------------------------
# 3b Paso 2a: crear la matriz V vacia, con valor NA (pag. 221)
# Dimension K x (N-1): una columna por cada tamanio muestral n=2,...,10
# ---------------------------------------------------------------
V <- matrix(NA, K, N - 1)

In [ ]:
# ---------------------------------------------------------------
# 3c Paso 2b: completar la matriz V con un doble ciclo for (pag. 221-222)
# Ciclo externo (k in 1:K): recorre las K muestras (filas)
# Ciclo interno (i in 2:N): recorre los tamanios muestrales (columnas)
# set.seed(k) asegura que los valores obtenidos sean replicables
# sum((x-mean(x))^2)/i calcula la varianza muestral sesgada
# ---------------------------------------------------------------
for (k in 1:K) {
  set.seed(k)
  for (i in 2:N) {
    x <- rnorm(i)
    V[k, (i - 1)] <- sum((x - mean(x))^2) / i
  }
}

In [ ]:
# ---------------------------------------------------------------
# 3d Paso 3: media aritmetica de cada columna con colMeans() (pag. 222)
# Como la varianza poblacional es sigma^2=1, el cociente
# observado sigma_barra_n^2 / sigma^2 es simplemente V.pond
# ---------------------------------------------------------------
V.pond <- colMeans(V)

print(V.pond)

In [ ]:
# ---------------------------------------------------------------
# 3e Pasos 4-5: Figura VI.3 - comparar con la prediccion teorica (n-1)/n (pag. 222)
# barplot(): diagrama de barras de los cocientes observados
# lines(): superpone la linea teorica (n-1)/n
# ---------------------------------------------------------------
barplot(V.pond, xlab = "Tamanio muestral (n)",
  ylim = c(0, 1), ylab = "Varianza muestral sesgada / Varianza poblacional",
  names.arg = 2:10, space = 0)
lines(((2:10) - 1) / 2:10, lwd = 4)
title("Sesgo de la varianza muestral")

# INTERPRETACION (pag. 222-223):
# El grafico corrobora que la varianza muestral sesgada converge segun
# la formula E[sigma_hat_*^2] = (n-1)/n * sigma^2. La primera columna
# (n=2) vale 0.5 = (2-1)/2*1, y a medida que n -> infinito el sesgo
# tiende asintoticamente a 0, ya que lim(n->inf) (n-1)/n = 1.

---
# 4. Aplicación empírica: análisis del teorema central del límite (pág. 223-235)

## 4.1. Definición (pág. 223-224)

El teorema central del límite (TCL) establece que si obtenemos una muestra aleatoria de una
población, entonces la distribución de la media muestral se distribuirá como una normal,
**independientemente de la distribución de la población original**.

Formalmente, si extraemos una muestra aleatoria simple de una distribución cualquiera con
media μ y desviación estándar σ, cada muestra Sₙ = X₁+...+Xₙ, y la media muestral (X̄) cumple:

X̄ ~ N(μ, σ/√n)

Además, a medida que n → ∞, la distribución de X̄ converge a una normal también en
**asimetría** (→ 0) y **curtosis** (→ 3).

## 4.2. Aproximación de una distribución totalmente aleatoria (pág. 224-229)

Empezamos creando datos aleatorios cuyo histograma no recuerde a ninguna distribución
conocida: 100 valores extraídos con reemplazo de la secuencia 1,...,100.

In [ ]:
# ---------------------------------------------------------------
# 4.2a Figura VI.4: poblacion totalmente aleatoria (pag. 224)
# sample(1:100, 100, replace=TRUE): 100 extracciones con reemplazo
# ---------------------------------------------------------------
set.seed(3)
pob <- sample(1:100, 100, replace = TRUE)
hist(pob, breaks = 80, col = "grey", main = "")

In [ ]:
# ---------------------------------------------------------------
# 4.2b Simular K=10.000 medias muestrales para n=4 (pag. 225)
# Doble estructura: se repite K veces la extraccion de una muestra
# de tamanio n y se guarda su media en el vector x.bar
# ---------------------------------------------------------------
n <- 4
K <- 10000

x.bar <- rep(NA, K)

for (i in 1:K) {
  set.seed(i)
  v <- sample(pob, n, replace = TRUE)
  x.bar[i] <- mean(v)
}

In [ ]:
# ---------------------------------------------------------------
# 4.2c Histograma de x.bar con densidad estimada superpuesta (pag. 226)
# lines(density(x.bar)): estimacion kernel de la densidad
# ---------------------------------------------------------------
hist(x.bar, breaks = 80, col = "grey", freq = FALSE,
  main = "n=4")
lines(density(x.bar), lwd = 4)

In [ ]:
# ---------------------------------------------------------------
# 4.2d Cargar la libreria moments: skewness() y kurtosis() (pag. 227)
# Necesaria para calcular los coeficientes de asimetria y curtosis
# ---------------------------------------------------------------
install.packages("moments")
library(moments)

In [ ]:
# ---------------------------------------------------------------
# 4.2e Ampliar el experimento: N=100 tamanios muestrales (pag. 227)
# m.x.bar: matriz K filas (muestras) x N columnas (tamanio muestral)
# Doble ciclo for: completa iterativamente la matriz de medias
# ---------------------------------------------------------------
N <- 100
m.x.bar <- matrix(NA, K, N)

for (i in 1:K) {
  set.seed(i)
  for (j in 1:N) {
    v <- sample(pob, j, replace = TRUE)
    m.x.bar[i, j] <- mean(v)
  }
}

In [ ]:
# ---------------------------------------------------------------
# 4.2f Estadisticos de cada columna con apply() (pag. 228)
# MARGIN=2: aplica la funcion a cada columna (cada tamanio muestral)
# ---------------------------------------------------------------
mu    <- apply(m.x.bar, 2, mean)
sigma <- apply(m.x.bar, 2, sd)
asim  <- apply(m.x.bar, 2, skewness)
curt  <- apply(m.x.bar, 2, kurtosis)

In [ ]:
# ---------------------------------------------------------------
# 4.2g Figura VI.6: grafico compuesto de los 4 estadisticos (pag. 228-229)
# par(mfrow=c(2,2)): 4 graficos en una cuadricula de 2 filas x 2 columnas
# abline(h=...): linea de referencia horizontal (valor teorico limite)
# dev.off(): restablece los parametros graficos por defecto
# ---------------------------------------------------------------
par(mfrow = c(2, 2))
plot(mu, type = "l", lwd = 3, main = "Media")
abline(h = mean(pob), lty = 2, lwd = 2)
plot(sigma, type = "l", lwd = 3,
  main = "Desviacion estandar")
plot(asim, type = "l", lwd = 3, main = "Asimetria")
abline(h = 0, lty = 2, lwd = 2)
plot(curt, type = "l", lwd = 3, main = "Curtosis")
abline(h = 3, lty = 2, lwd = 2)
dev.off()

# INTERPRETACION (pag. 229):
# La tendencia de los cuatro estadisticos confirma el TCL:
# X.barra ~ N(mu, sigma/sqrt(n)). La media converge a mu, la
# desviacion estandar decrece como sigma/sqrt(n), la asimetria
# tiende a 0 y la curtosis tiende a 3.

## 4.3. Aproximación de diferentes distribuciones a la normal (pág. 230-235)

Para generalizar la simulación anterior, se «empaqueta» todo el proceso en una función propia
`tcl(x)`, que recibe un vector numérico `x` (la población) y reproduce automáticamente el
experimento completo: extrae K=10.000 muestras para cada tamaño n=1,...,100, calcula los
cuatro estadísticos y los representa en un gráfico compuesto.

In [ ]:
# ---------------------------------------------------------------
# 4.3a Funcion propia tcl(x): empaqueta toda la simulacion (pag. 230-231)
# Parametro: x (vector numerico con la poblacion de origen)
# Efecto: genera el grafico compuesto de Media/Desv.est/Asimetria/Curtosis
# ---------------------------------------------------------------
tcl <- function(x) {
  library(moments)
  K <- 10000
  N <- 100
  m.x.bar <- matrix(NA, K, N)
  for (i in 1:K) {
    set.seed(i)
    for (j in 1:N) {
      v <- sample(x, j, replace = TRUE)
      m.x.bar[i, j] <- mean(v)
    }
  }
  mu <- apply(m.x.bar, 2, mean)
  sigma <- apply(m.x.bar, 2, sd)
  asim <- apply(m.x.bar, 2, skewness)
  curt <- apply(m.x.bar, 2, kurtosis)
  par(mfrow = c(2, 2))
  plot(mu, type = "l", lwd = 3, main = "Media")
  abline(h = mean(x), lty = 2, lwd = 2)
  plot(sigma, type = "l", lwd = 3,
  main = "Desviacion estandar")
  plot(asim, type = "l", lwd = 3, main = "Asimetria")
  abline(h = 0, lty = 2, lwd = 2)
  plot(curt, type = "l", lwd = 3, main = "Curtosis")
  abline(h = 3, lty = 2, lwd = 2)
}

In [ ]:
# ---------------------------------------------------------------
# 4.3b Figura VI.7: poblacion Poisson(lambda=1) (pag. 231)
# rpois(n, lambda) genera n valores aleatorios de una Poisson
# ---------------------------------------------------------------
set.seed(5)
pob.2 <- rpois(10000, 1)
hist(pob.2, col = "grey", main = "")

In [ ]:
# ---------------------------------------------------------------
# 4.3c Coeficientes de asimetria y curtosis de pob.2 (pag. 232)
# La distribucion de Poisson(1) es asimetrica positiva y leptocurtica
# (curtosis > 3)
# ---------------------------------------------------------------
cbind(skewness(pob.2), kurtosis(pob.2))
#            [,1]     [,2]
# [1,] 0.9991975 3.914486

In [ ]:
# ---------------------------------------------------------------
# 4.3d Figura VI.8: aplicar tcl() a la poblacion Poisson(1) (pag. 233)
# A medida que n -> infinito, la media fluctua menos y tiende al
# valor poblacional; asimetria y curtosis parten de 0.999 y 3.914
# y se aproximan a los valores normales 0 y 3.
# ---------------------------------------------------------------
tcl(pob.2)

In [ ]:
# ---------------------------------------------------------------
# 4.3e Figura VI.9: poblacion Poisson(lambda=100) (pag. 233-234)
# Resultado del TCL: si X ~ Poisson(lambda) con lambda>10,
# entonces aproximadamente X ~ N(lambda, lambda)
# ---------------------------------------------------------------
set.seed(5)
pob.3 <- rpois(10000, 100)
hist(pob.3, col = "grey")

In [ ]:
# ---------------------------------------------------------------
# 4.3f Coeficientes de asimetria y curtosis de pob.3 (pag. 234)
# Al ser lambda=100 grande, la Poisson ya es casi normal de origen
# ---------------------------------------------------------------
cbind(skewness(pob.3), kurtosis(pob.3))
#            [,1]     [,2]
# [1,] 0.09443229 2.969321

In [ ]:
# ---------------------------------------------------------------
# 4.3g Figura VI.10: aplicar tcl() a la poblacion Poisson(100) (pag. 235)
# Los coeficientes de asimetria y curtosis ya son aproximadamente
# normales incluso para n pequenio, y se mantienen asi cuando n -> infinito
# ---------------------------------------------------------------
tcl(pob.3)

---
## Conclusión

Este notebook cubre la totalidad del código R del **Capítulo VI** de Liviano & Pujol (2017),
siguiendo el mismo orden y numeración de secciones que el texto original.

| Sección del libro | Tema | Funciones clave |
|---|---|---|
| 1 | Motivación | — |
| 2 | Generación de valores aleatorios | `set.seed()`, `sample()`, `rnorm()`, `rexp()`, `curve()` |
| 3 | Sesgo del estimador de la varianza | bucle `for` doble, `colMeans()`, `barplot()`, `lines()` |
| 4.1 | Definición del TCL | — |
| 4.2 | TCL sobre población aleatoria | `apply()`, `skewness()`, `kurtosis()` (paquete `moments`), `par(mfrow=)` |
| 4.3 | Función propia `tcl()` y Poisson | `rpois()`, función definida por el usuario |

### Cómo ejecutar
1. Abrir en **Jupyter Lab**, **VS Code** o **Google Colab** con kernel `R`.
2. Ejecutar las celdas en orden: la sección 4 reutiliza `pob` (creado en 4.2) y la función `tcl()`
   (definida en 4.3a) sobre `pob.2` y `pob.3`.
3. La sección 4 requiere el paquete `moments` (se instala automáticamente).
4. Las simulaciones con K=100.000 (sección 3) y K=10.000 × N=100 (sección 4.2) son
   computacionalmente intensivas y pueden tardar varios minutos.

---
*Notebook reescrito en R a partir del PDF oficial del capítulo (Cap06_Programacion_Simulacion_R.pdf), sustituyendo la versión previa en Python.*
*Referencia: Liviano Solís, D. & Pujol Jover, M. (2017). Análisis cuantitativo con R: matemáticas, estadística y econometría. Editorial UOC.*